# 1. Ingestion and data quality

Three internal HR extracts, one central table. This notebook builds it and checks it, and
the checks are the point: everything downstream reads the file it writes — the exploratory
work, the model, the service — so an error made here is an error nobody downstream can see.

The reusable steps live in `src/attrition_serving/`; the notebook orchestrates them. The
counts it prints are also written to `reports/` by
`uv run python scripts/data_quality_report.py`, so a reader who does not run the notebook
still gets the numbers.


## The data

Three sources, joined on the employee:

1. **HRIS**: contract, demographics, role, tenure
2. **Annual reviews**: performance and satisfaction ratings
3. **Internal survey**: perceived wellbeing, and the target, whether the employee left

Before anything is modelled, four things have to be true: the structure of each source is
understood, the quality is measured rather than assumed, the join keys are known to be
unique, and the identifiers are anonymised.


## How this notebook proceeds

1. Load the three raw CSVs
2. Check them — shape, types, missing values, duplicates
3. Harmonise the join keys
4. Join
5. Check again, because a join is where rows silently multiply
6. Anonymise the employee identifiers
7. Save the central table

`reports/cleaning_trace.csv` records steps 2, 4 and 7 side by side, so what the join did to
the row count is visible instead of assumed.


In [1]:
from attrition_serving.config import PATHS
from attrition_serving.data.io import (
    anonymize_employee_id,
    check_duplicates,
    join_sources,
    load_eval,
    load_sirh,
    load_sondage,
)

## Loading the raw extracts

The CSVs are in `data/raw/`, exactly as they were exported. Nothing is transformed here —
the point of a separate load step is to know what arrived before deciding what to do
with it.


In [2]:
sirh_path = PATHS.data_raw / "extrait_sirh.csv"
eval_path = PATHS.data_raw / "extrait_eval.csv"
sondage_path = PATHS.data_raw / "extrait_sondage.csv"

sirh_df = load_sirh(sirh_path)
eval_df = load_eval(eval_path)
sondage_df = load_sondage(sondage_path)

sirh_df.shape, eval_df.shape, sondage_df.shape

((1470, 12), (1470, 11), (1470, 12))

## First look

For each source: shape, dtypes, and the first rows. What this catches early is a column
read as text because one row holds a stray character, and a column that is entirely
constant — both of which are invisible once the sources are joined.


In [3]:
sirh_df.head()

,id_employee,age,genre,revenu_mensuel,statut_marital,departement,poste,nombre_experiences_precedentes,nombre_heures_travailless,annee_experience_totale,annees_dans_l_entreprise,annees_dans_le_poste_actuel
0,1,41,F,5993,Célibataire,Commercial,Cadre Commercial,8,80,8,6,4
1,2,49,M,5130,Marié(e),Consulting,Assistant de Direction,1,80,10,10,7
2,4,37,M,2090,Célibataire,Consulting,Consultant,6,80,7,0,0
3,5,33,F,2909,Marié(e),Consulting,Assistant de Direction,1,80,8,8,7
4,7,27,M,3468,Marié(e),Consulting,Consultant,9,80,6,2,2


In [4]:
sirh_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 12 columns):
 #   Column                          Non-Null Count  Dtype 
---  ------                          --------------  ----- 
 0   id_employee                     1470 non-null   int64 
 1   age                             1470 non-null   int64 
 2   genre                           1470 non-null   object
 3   revenu_mensuel                  1470 non-null   int64 
 4   statut_marital                  1470 non-null   object
 5   departement                     1470 non-null   object
 6   poste                           1470 non-null   object
 7   nombre_experiences_precedentes  1470 non-null   int64 
 8   nombre_heures_travailless       1470 non-null   int64 
 9   annee_experience_totale         1470 non-null   int64 
 10  annees_dans_l_entreprise        1470 non-null   int64 
 11  annees_dans_le_poste_actuel     1470 non-null   int64 
dtypes: int64(8), object(4)
memory usage: 137.9+ KB


## Duplicates

A join multiplies rows when a key is not unique, and it does it quietly: the result has
more rows than either input and every one of them looks plausible.

Two checks, before joining anything: exact duplicate rows, and duplicate values on each
candidate key.


In [5]:
check_duplicates(sirh_df, subset=["id_employee"]).shape

(0, 12)

In [6]:
check_duplicates(eval_df, subset=["eval_number"]).shape

(0, 11)

In [7]:
check_duplicates(sondage_df, subset=["code_sondage"]).shape

(0, 12)

## Harmonising the join keys

The three sources do not share a key:

- **HRIS**: `id_employee`, an integer
- **Reviews**: `eval_number`, a string like `E_1`
- **Survey**: `code_sondage`, an integer

The numeric part of `eval_number` is extracted into `eval_number_int` at load time, in the
loader rather than here, so every consumer of these files gets the same key.

Both keys survive into the feature table and neither is used by the model — they are
identifiers. `reports/redundant_features.csv` reports them as perfectly correlated, which
is what two names for the same thing look like.


In [8]:
eval_df[["eval_number", "eval_number_int"]].head()

,eval_number,eval_number_int
0,E_1,1
1,E_2,2
2,E_4,4
3,E_5,5
4,E_7,7


## Joining

An **inner** join: only employees present in all three systems are kept.

That is a choice, and it costs coverage. It is the right one here because the target comes
from the survey and the features from the other two, so a row missing from any source is a
row that cannot be used for either training or evaluation. Keeping it would mean imputing
the very thing being predicted.


In [9]:
df = join_sources(sirh_df, eval_df, sondage_df, how="inner")
df.shape

(1470, 35)

## Checking the join

Three things, in order of how badly they would mislead:

- the row count is unchanged, so no key multiplied;
- no unexpected duplicate appeared;
- the target is present and its distribution is what the survey said it was.


In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df.columns

Index(['id_employee', 'age', 'genre', 'revenu_mensuel', 'statut_marital',
       'departement', 'poste', 'nombre_experiences_precedentes',
       'nombre_heures_travailless', 'annee_experience_totale',
       'annees_dans_l_entreprise', 'annees_dans_le_poste_actuel',
       'satisfaction_employee_environnement', 'note_evaluation_precedente',
       'niveau_hierarchique_poste', 'satisfaction_employee_nature_travail',
       'satisfaction_employee_equipe',
       'satisfaction_employee_equilibre_pro_perso', 'eval_number',
       'note_evaluation_actuelle', 'heure_supplementaires',
       'augementation_salaire_precedente', 'eval_number_int',
       'a_quitte_l_entreprise', 'nombre_participation_pee',
       'nb_formations_suivies', 'nombre_employee_sous_responsabilite',
       'code_sondage', 'distance_domicile_travail', 'niveau_education',
       'domaine_etude', 'ayant_enfants', 'frequence_deplacement',
       'annees_depuis_la_derniere_promotion', 'annes_sous_responsable_actuel'],
   

## Anonymising the identifiers

HR data identifies people, and an employee identifier is enough to re-identify one.

**HMAC-SHA256 with a secret key.** Stable, so the same employee keeps the same anonymised
id across runs and the join still works; not reversible without the key, so the file can be
handled without carrying the roster with it.

The key lives in `ANONYMIZATION_KEY` and never in the repository. Anonymisation with a
committed key is a slower way of publishing the identifiers.


In [12]:
df["employee_id_anon"] = anonymize_employee_id(df["id_employee"])
df = df.drop(columns=["id_employee"])

df[["employee_id_anon"]].head()

,employee_id_anon
0,emp_338e593c7e90a79d
1,emp_89f1c49e9609aabf
2,emp_8b1641b2cb8432db
3,emp_6fac8ca2e88ae968
4,emp_68169193b4301586


## Saving the central table

Written to `data/processed/`. This file is the single input to everything after it: the
exploratory notebook, the feature engineering, the SQL views, and the model.

The raw extracts and this table are **not** committed. They are HR records about real
people, and a portfolio repository is a public place; `data/` holds only the ten-row request
fixtures the API tests need.


In [13]:
output_path = PATHS.data_processed / "employees_joined.parquet"
df.to_parquet(output_path, index=False)

output_path

WindowsPath('C:/Users/Ben/AppData/Local/attrition-data/processed/employees_joined.parquet')

## What this step produced

A central table from three sources, checked at each stage, with identifiers anonymised.

`reports/cleaning_trace.csv` records what the checks found: the three extracts are complete
— no missing cells, no duplicate rows, 1,470 rows each — and they join one-to-one. The
twenty-seven missing values in the final table are **created** by the feature engineering
that follows, not inherited from the sources. That distinction matters when deciding how to
impute them: they come from divisions whose denominator can be zero, not from unanswered
questions.
